# T Cell Analysis: BBKNN Integration + Wilcoxon Marker Finding

**Purpose**: Re-integrate T cells by dataset using BBKNN, then identify marker genes

**Workflow**:
1. Load T cell data (post-cNMF)
2. **Filter out epithelial contamination from scanvi_predict**
3. Inspect dataset distribution and filter small datasets
4. Re-run preprocessing with batch-aware HVG
5. BBKNN batch correction by dataset
6. Clustering and visualization
7. Visualize T cell canonical markers
8. Wilcoxon differential expression
9. Export results

**Author**: r2end  
**Date**: 2025-01-07  
**Update**: Added epithelial cell filtering step

## Configuration

In [ ]:
# ============================================================================
# CONFIGURATION PARAMETERS
# ============================================================================

# File paths
INPUT_H5AD = "/home/h2048/data/py/1217/cnmf_batch_production_v1_1_1/T_cells/batch_aware/cnmf_analysis_k40_1/T_cells_with_cnmf_k40.h5ad"
OUTPUT_DIR = "/home/h2048/data/R/0107/t_bbknn_filtered/"

# Epithelial filtering
SCANVI_COLUMN = 'scanvi_prediction'  # Column name for scanvi predictions
# Keywords to identify epithelial cells (case-insensitive matching)
EPITHELIAL_KEYWORDS = ['epithelial', 'basal', 'goblet', 'ciliated', 'secretory', 'club']

# Dataset filtering
BATCH_KEY = 'dataset'           # Column name for dataset/batch
MIN_CELLS_PER_DATASET = 20      # Filter datasets with < N cells

# Preprocessing
N_TOP_GENES = 4000              # Number of highly variable genes
N_PCS = 50                      # Number of PCs

# BBKNN parameters
BBKNN_NEIGHBORS_WITHIN_BATCH = 5
BBKNN_N_PCS = 50
BBKNN_TRIM = 30

# Clustering
LEIDEN_RESOLUTION = 1

# Differential expression
MIN_LOGFC = 0.25
MIN_PCT = 0.1
TOP_N_MARKERS = 50

# Visualization
FIGURE_DPI = 300
FIGURE_FORMAT = 'pdf'
UMAP_SIZE = 3

# Performance
N_JOBS = 8

print("✓ Configuration loaded")

## T Cell Canonical Markers

Define key marker genes for T cell subtypes for validation

In [ ]:
# ============================================================================
# T CELL MARKER GENES
# ============================================================================

# Core T cell markers
TCELL_CORE_MARKERS = {
    'Pan_T': ['CD3D', 'CD3E', 'CD3G'],
    'CD4_T': ['CD4', 'CD40LG'],
    'CD8_T': ['CD8A', 'CD8B'],
    'NK': ['GNLY', 'NKG7', 'KLRD1', 'FCGR3A'],
    'Naive': ['CCR7', 'TCF7', 'LEF1', 'SELL', 'IL7R'],
    'Memory': ['GZMK', 'CD69'],
    'Effector': ['GZMB', 'GZMH', 'PRF1', 'IFNG'],
    'Treg': ['FOXP3', 'IL2RA', 'IKZF2', 'TNFRSF4'],
    'Exhausted': ['PDCD1', 'HAVCR2', 'LAG3', 'TIGIT'],
    'Proliferating': ['MKI67', 'TOP2A', 'STMN1'],
    'Th2': ['GATA3', 'IL4', 'IL5', 'IL13'],
    'Th17': ['RORC', 'IL17A', 'IL23R'],
}

# Flat list for quick visualization
TCELL_KEY_MARKERS = [
    'CD3D', 'CD4', 'CD8A',           # Major lineages
    'GNLY', 'NKG7',                  # NK
    'CCR7', 'SELL', 'IL7R',          # Naive
    'GZMK', 'CD69',                  # Memory
    'GZMB', 'PRF1',                  # Effector
    'FOXP3', 'IL2RA',                # Treg
    'PDCD1', 'HAVCR2',               # Exhausted
    'MKI67'                          # Proliferating
]

print("T cell marker genes defined:")
for category, markers in TCELL_CORE_MARKERS.items():
    print(f"  {category}: {', '.join(markers)}")

## Imports & Setup

In [ ]:
# ============================================================================
# IMPORTS
# ============================================================================

import scanpy as sc
import scanpy.external as sce
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import sparse
import warnings
warnings.filterwarnings('ignore')

# Scanpy settings
sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=FIGURE_DPI, facecolor='white', frameon=False)
sc.settings.n_jobs = N_JOBS

# Create output directories
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)
fig_dir = output_dir / "figures"
fig_dir.mkdir(exist_ok=True)

print("="*80)
print("T Cell BBKNN + Marker Analysis (Epithelial Filtered)")
print("="*80)
print(f"Output directory: {output_dir}")

## Step 1: Load Data & Inspect

In [ ]:
# ============================================================================
# STEP 1: DATA LOADING
# ============================================================================

print("\n" + "="*80)
print("STEP 1: DATA LOADING & INSPECTION")
print("="*80)

print(f"\nLoading: {INPUT_H5AD}")
adata = sc.read_h5ad(INPUT_H5AD)

print(f"\nData dimensions (before filtering):")
print(f"  Cells: {adata.n_obs:,}")
print(f"  Genes: {adata.n_vars:,}")

# Check metadata
print(f"\nAvailable .obs columns:")
for col in adata.obs.columns:
    n_unique = adata.obs[col].nunique()
    print(f"  - {col}: {n_unique} unique values")

# Check if cNMF results exist
cnmf_columns = [col for col in adata.obs.columns if 'cnmf' in col.lower() or 'usage' in col.lower()]
if len(cnmf_columns) > 0:
    print(f"\n✓ cNMF results detected:")
    for col in cnmf_columns[:5]:  # Show first 5
        print(f"    {col}")
    print(f"  (and {len(cnmf_columns)-5} more...)" if len(cnmf_columns) > 5 else "")
else:
    print(f"\n⚠️  No cNMF results found")

# Check batch key
if BATCH_KEY not in adata.obs.columns:
    raise ValueError(f"Batch key '{BATCH_KEY}' not found!")

## Step 2: Filter Epithelial Contamination

Remove cells annotated as epithelial by scanvi_predict to ensure pure T/NK cell population

In [ ]:
# ============================================================================
# STEP 2: FILTER EPITHELIAL CELLS
# ============================================================================

print("\n" + "="*80)
print("STEP 2: EPITHELIAL CONTAMINATION FILTERING")
print("="*80)

# Check if scanvi_predict column exists
if SCANVI_COLUMN not in adata.obs.columns:
    print(f"\n⚠️  Warning: '{SCANVI_COLUMN}' column not found!")
    print(f"Available columns: {', '.join(adata.obs.columns)}")
    print(f"\nSkipping epithelial filtering...")
else:
    # Show cell type distribution before filtering
    print(f"\nCell type distribution (before filtering):")
    celltypes_before = adata.obs[SCANVI_COLUMN].value_counts()
    for ct, count in celltypes_before.head(20).items():
        print(f"  {ct}: {count:,} cells")
    if len(celltypes_before) > 20:
        print(f"  ... and {len(celltypes_before)-20} more cell types")
    
    # Identify epithelial cells (case-insensitive)
    epithelial_mask = adata.obs[SCANVI_COLUMN].str.lower().apply(
        lambda x: any(keyword in str(x).lower() for keyword in EPITHELIAL_KEYWORDS)
    )
    
    n_epithelial = epithelial_mask.sum()
    pct_epithelial = 100 * n_epithelial / len(adata)
    
    print(f"\nEpithelial contamination detected:")
    print(f"  Keywords used: {', '.join(EPITHELIAL_KEYWORDS)}")
    print(f"  Epithelial cells: {n_epithelial:,} ({pct_epithelial:.2f}%)")
    
    if n_epithelial > 0:
        # Show which epithelial types were found
        epithelial_types = adata.obs.loc[epithelial_mask, SCANVI_COLUMN].value_counts()
        print(f"\nEpithelial subtypes to be removed:")
        for etype, count in epithelial_types.items():
            print(f"  - {etype}: {count:,} cells")
        
        # Filter out epithelial cells
        print(f"\nRemoving epithelial cells...")
        adata = adata[~epithelial_mask].copy()
        
        print(f"\n✓ Filtering complete")
        print(f"  Cells after filtering: {adata.n_obs:,}")
        print(f"  Cells removed: {n_epithelial:,} ({pct_epithelial:.2f}%)")
        
        # Show remaining cell type distribution
        print(f"\nRemaining cell type distribution:")
        celltypes_after = adata.obs[SCANVI_COLUMN].value_counts()
        for ct, count in celltypes_after.head(15).items():
            print(f"  {ct}: {count:,} cells")
        if len(celltypes_after) > 15:
            print(f"  ... and {len(celltypes_after)-15} more cell types")
    else:
        print(f"\n✓ No epithelial contamination detected")

## Step 3: Dataset Distribution & Small Batch Filtering

In [ ]:
# ============================================================================
# STEP 3: DATASET FILTERING
# ============================================================================

print("\n" + "="*80)
print("STEP 3: DATASET DISTRIBUTION & FILTERING")
print("="*80)

# Dataset distribution
print(f"\nDataset distribution ({BATCH_KEY}):")
dataset_counts = adata.obs[BATCH_KEY].value_counts().sort_values(ascending=False)
for dataset, count in dataset_counts.items():
    print(f"  {dataset}: {count:,} cells")

# Filter small datasets
small_datasets = dataset_counts[dataset_counts < MIN_CELLS_PER_DATASET].index
if len(small_datasets) > 0:
    print(f"\n⚠️  Found {len(small_datasets)} small datasets (< {MIN_CELLS_PER_DATASET} cells):")
    for ds in small_datasets:
        print(f"    - {ds}: {dataset_counts[ds]} cells")
    
    print(f"\nRemoving small datasets...")
    n_cells_before = adata.n_obs
    adata = adata[~adata.obs[BATCH_KEY].isin(small_datasets)].copy()
    n_cells_removed = n_cells_before - adata.n_obs
    
    print(f"  ✓ Removed {n_cells_removed:,} cells from {len(small_datasets)} small datasets")
    print(f"  Remaining cells: {adata.n_obs:,}")
    print(f"  Remaining datasets: {adata.obs[BATCH_KEY].nunique()}")
else:
    print(f"\n✓ All datasets have >= {MIN_CELLS_PER_DATASET} cells")

print(f"\nFinal dataset distribution:")
final_counts = adata.obs[BATCH_KEY].value_counts().sort_values(ascending=False)
for dataset, count in final_counts.items():
    print(f"  {dataset}: {count:,} cells")

## Step 4: Preprocessing

In [ ]:
# ============================================================================
# STEP 4: PREPROCESSING
# ============================================================================

print("\n" + "="*80)
print("STEP 4: PREPROCESSING")
print("="*80)

# Check data layers
print(f"\nAvailable layers:")
if adata.layers is not None and len(adata.layers) > 0:
    for layer in adata.layers.keys():
        print(f"  - {layer}")
else:
    print(f"  No layers found")

# Ensure we have counts in a layer
if 'counts' not in adata.layers:
    print(f"\n⚠️  'counts' layer not found, creating from .X...")
    # Assuming .X is raw counts or can be used
    adata.layers['counts'] = adata.X.copy()

print(f"\nNormalizing and log-transforming...")
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
print(f"  ✓ Normalized to 10,000 counts per cell + log1p")

# HVG selection with batch awareness
print(f"\nSelecting highly variable genes (n={N_TOP_GENES})...")
try:
    sc.pp.highly_variable_genes(
        adata,
        n_top_genes=N_TOP_GENES,
        batch_key=BATCH_KEY,
        subset=False
    )
    hvg_method = "batch-aware"
    print(f"  ✓ Batch-aware HVG selection successful")
except Exception as e:
    print(f"  ⚠️  Batch-aware HVG failed: {e}")
    print(f"  Falling back to standard HVG...")
    sc.pp.highly_variable_genes(
        adata,
        n_top_genes=N_TOP_GENES,
        subset=False
    )
    hvg_method = "standard"
    print(f"  ✓ Standard HVG selection complete")

n_hvg = adata.var['highly_variable'].sum()
print(f"  HVGs selected: {n_hvg:,}")
print(f"  Method: {hvg_method}")

# Save full gene set to .raw before subsetting
print(f"\nSaving full gene set to .raw...")
adata.raw = adata.copy()
print(f"  ✓ Full {adata.raw.n_vars:,} genes preserved in .raw")

# Subset to HVGs
print(f"\nSubsetting to HVGs...")
adata = adata[:, adata.var['highly_variable']].copy()
print(f"  ✓ Working dataset: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

## Step 5: PCA

In [ ]:
# ============================================================================
# STEP 5: PCA
# ============================================================================

print("\n" + "="*80)
print("STEP 5: PCA")
print("="*80)

print(f"\nScaling data...")
sc.pp.scale(adata, max_value=10)
print(f"  ✓ Scaled (max_value=10)")

print(f"\nRunning PCA (n_comps={N_PCS})...")
sc.tl.pca(adata, n_comps=N_PCS, svd_solver='arpack')
print(f"  ✓ PCA complete")

# Variance explained
var_ratio = adata.uns['pca']['variance_ratio']
cum_var = np.cumsum(var_ratio)
print(f"\nVariance explained:")
print(f"  PC1-10: {cum_var[9]:.1%}")
print(f"  PC1-30: {cum_var[29]:.1%}")
print(f"  PC1-50: {cum_var[49]:.1%}")

## Step 6: BBKNN Batch Correction

In [ ]:
# ============================================================================
# STEP 6: BBKNN BATCH CORRECTION
# ============================================================================

print("\n" + "="*80)
print("STEP 6: BBKNN BATCH CORRECTION")
print("="*80)

print(f"\nRunning BBKNN...")
print(f"  Parameters:")
print(f"    - batch_key: {BATCH_KEY}")
print(f"    - neighbors_within_batch: {BBKNN_NEIGHBORS_WITHIN_BATCH}")
print(f"    - n_pcs: {BBKNN_N_PCS}")
print(f"    - trim: {BBKNN_TRIM}")

sce.pp.bbknn(
    adata,
    batch_key=BATCH_KEY,
    neighbors_within_batch=BBKNN_NEIGHBORS_WITHIN_BATCH,
    n_pcs=BBKNN_N_PCS,
    trim=BBKNN_TRIM
)

print(f"  ✓ BBKNN complete")
print(f"  Neighbor graph constructed")

## Step 7: UMAP & Clustering

In [ ]:
# ============================================================================
# STEP 7: UMAP & CLUSTERING
# ============================================================================

print("\n" + "="*80)
print("STEP 7: UMAP & CLUSTERING")
print("="*80)

print(f"\nRunning UMAP...")
sc.tl.umap(adata)
print(f"  ✓ UMAP complete")

print(f"\nRunning Leiden clustering (resolution={LEIDEN_RESOLUTION})...")
sc.tl.leiden(adata, resolution=LEIDEN_RESOLUTION, key_added='leiden')
n_clusters = adata.obs['leiden'].nunique()
print(f"  ✓ Leiden complete")
print(f"  Clusters identified: {n_clusters}")

# Cluster sizes
print(f"\nCluster sizes:")
cluster_counts = adata.obs['leiden'].value_counts().sort_index()
for cluster, count in cluster_counts.items():
    pct = 100 * count / len(adata)
    print(f"  Cluster {cluster}: {count:,} cells ({pct:.1f}%)")

## Step 8: Visualization - Clustering & Batch

In [ ]:
# ============================================================================
# STEP 8: VISUALIZATION - CLUSTERING & BATCH
# ============================================================================

print("\n" + "="*80)
print("STEP 8: VISUALIZATION - CLUSTERING & BATCH")
print("="*80)

# UMAP by clusters
print(f"\nPlotting UMAP by clusters...")
fig = sc.pl.umap(
    adata,
    color='leiden',
    title='T Cell Clusters (BBKNN)',
    legend_loc='on data',
    legend_fontsize=8,
    size=UMAP_SIZE,
    frameon=False,
    show=False,
    return_fig=True
)
fig.savefig(fig_dir / f'01_umap_clusters.{FIGURE_FORMAT}', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print(f"  ✓ Saved: 01_umap_clusters.{FIGURE_FORMAT}")

# UMAP by batch
print(f"\nPlotting UMAP by batch...")
fig = sc.pl.umap(
    adata,
    color=BATCH_KEY,
    title='T Cells by Dataset (BBKNN)',
    size=UMAP_SIZE,
    frameon=False,
    show=False,
    return_fig=True
)
fig.savefig(fig_dir / f'02_umap_batch.{FIGURE_FORMAT}', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print(f"  ✓ Saved: 02_umap_batch.{FIGURE_FORMAT}")

# Combined plot
print(f"\nPlotting combined UMAP...")
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sc.pl.umap(adata, color='leiden', ax=axes[0], show=False, title='Clusters', size=UMAP_SIZE, frameon=False)
sc.pl.umap(adata, color=BATCH_KEY, ax=axes[1], show=False, title='Dataset', size=UMAP_SIZE, frameon=False)

plt.tight_layout()
plt.savefig(fig_dir / f'03_umap_combined.{FIGURE_FORMAT}', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print(f"  ✓ Saved: 03_umap_combined.{FIGURE_FORMAT}")

## Step 9: T Cell Marker Visualization

In [ ]:
# ============================================================================
# STEP 9: T CELL MARKER VISUALIZATION
# ============================================================================

print("\n" + "="*80)
print("STEP 9: T CELL MARKER VISUALIZATION")
print("="*80)

# Check which markers are available
available_markers = [m for m in TCELL_KEY_MARKERS if m in adata.raw.var_names]
missing_markers = [m for m in TCELL_KEY_MARKERS if m not in adata.raw.var_names]

print(f"\nMarker availability:")
print(f"  Available: {len(available_markers)}/{len(TCELL_KEY_MARKERS)}")
if len(missing_markers) > 0:
    print(f"  Missing: {', '.join(missing_markers)}")

# UMAP with T cell markers
if len(available_markers) > 0:
    print(f"\nPlotting T cell markers on UMAP...")
    fig = sc.pl.umap(
        adata,
        color=available_markers,
        use_raw=True,
        ncols=4,
        size=UMAP_SIZE,
        frameon=False,
        vmax='p99',
        cmap='viridis',
        show=False,
        return_fig=True
    )
    fig.savefig(fig_dir / f'04_tcell_markers_umap.{FIGURE_FORMAT}', dpi=FIGURE_DPI, bbox_inches='tight')
    plt.show()
    print(f"  ✓ Saved: 04_tcell_markers_umap.{FIGURE_FORMAT}")
    
    # Dotplot
    print(f"\nGenerating dotplot...")
    fig = sc.pl.dotplot(
        adata,
        var_names=available_markers,
        groupby='leiden',
        use_raw=True,
        dendrogram=True,
        show=False,
        return_fig=True
    )
    fig.savefig(fig_dir / f'05_tcell_markers_dotplot.{FIGURE_FORMAT}', dpi=FIGURE_DPI, bbox_inches='tight')
    plt.show()
    print(f"  ✓ Saved: 05_tcell_markers_dotplot.{FIGURE_FORMAT}")
else:
    print(f"\n⚠️  No T cell markers available for visualization")

## Step 10: Wilcoxon Differential Expression

In [ ]:
# ============================================================================
# STEP 10: WILCOXON DIFFERENTIAL EXPRESSION
# ============================================================================

print("\n" + "="*80)
print("STEP 10: WILCOXON DIFFERENTIAL EXPRESSION")
print("="*80)

print(f"\nRunning Wilcoxon rank-sum test...")
print(f"  Parameters:")
print(f"    - min_logfc: {MIN_LOGFC}")
print(f"    - min_pct: {MIN_PCT}")

sc.tl.rank_genes_groups(
    adata,
    groupby='leiden',
    method='wilcoxon',
    use_raw=True,
    key_added='rank_genes_wilcox',
    pts=True
)

print(f"  ✓ Wilcoxon test complete")

# Extract and filter results
print(f"\nExtracting marker genes...")
all_markers = []

for cluster in adata.obs['leiden'].cat.categories:
    cluster_result = sc.get.rank_genes_groups_df(
        adata,
        group=cluster,
        key='rank_genes_wilcox'
    )
    
    # Filter by logFC and adjusted p-value
    cluster_result_filtered = cluster_result[
        (cluster_result['logfoldchanges'].abs() > MIN_LOGFC) &
        (cluster_result['pvals_adj'] < 0.05)
    ].head(TOP_N_MARKERS)
    
    # Calculate percent expressed
    cluster_mask = adata.obs['leiden'] == cluster
    other_mask = ~cluster_mask
    
    cluster_cells = adata.raw.X[cluster_mask]
    other_cells = adata.raw.X[other_mask]
    
    pct_in = []
    pct_out = []
    
    for gene in cluster_result_filtered['names']:
        gene_idx = adata.raw.var_names.get_loc(gene)
        
        if sparse.issparse(cluster_cells):
            expr_in = cluster_cells[:, gene_idx].toarray().flatten()
            expr_out = other_cells[:, gene_idx].toarray().flatten()
        else:
            expr_in = cluster_cells[:, gene_idx].flatten()
            expr_out = other_cells[:, gene_idx].flatten()
        
        pct_in.append(np.sum(expr_in > 0) / len(expr_in))
        pct_out.append(np.sum(expr_out > 0) / len(expr_out))
    
    cluster_result_filtered['cluster'] = cluster
    cluster_result_filtered['pct_in_cluster'] = pct_in
    cluster_result_filtered['pct_out_cluster'] = pct_out
    
    cluster_result_filtered = cluster_result_filtered[
        cluster_result_filtered['pct_in_cluster'] > MIN_PCT
    ]
    
    n_markers = len(cluster_result_filtered)
    print(f"  Cluster {cluster}: {n_markers} markers")
    
    all_markers.append(cluster_result_filtered)

# Combine results
if len(all_markers) > 0:
    markers_df = pd.concat(all_markers, ignore_index=True)
    markers_df = markers_df.sort_values(['cluster', 'pvals_adj'])
    
    markers_df.to_csv(output_dir / "tcell_markers_wilcox_all.csv", index=False)
    print(f"\n✓ Total markers: {len(markers_df):,}")
    print(f"✓ Saved: tcell_markers_wilcox_all.csv")
    
    top_markers = markers_df.groupby('cluster').head(10)
    top_markers.to_csv(output_dir / "tcell_markers_wilcox_top10.csv", index=False)
    print(f"✓ Saved: tcell_markers_wilcox_top10.csv")
else:
    print(f"\n⚠️  No markers passed filtering!")
    markers_df = pd.DataFrame()

## Step 11: Marker Visualization

In [ ]:
# ============================================================================
# STEP 11: MARKER VISUALIZATION
# ============================================================================

if len(markers_df) > 0:
    print("\n" + "="*80)
    print("STEP 11: MARKER VISUALIZATION")
    print("="*80)
    
    # Heatmap
    print(f"\nGenerating heatmap (top 5 per cluster)...")
    try:
        fig = sc.pl.rank_genes_groups_heatmap(
            adata,
            n_genes=5,
            key='rank_genes_wilcox',
            use_raw=True,
            show=False,
            cmap='RdBu_r',
            figsize=(12, 10),
            vmin=-3,
            vmax=3,
            dendrogram=False
        )
        plt.savefig(fig_dir / f'06_marker_heatmap.{FIGURE_FORMAT}', dpi=FIGURE_DPI, bbox_inches='tight')
        plt.show()
        print(f"  ✓ Saved: 06_marker_heatmap.{FIGURE_FORMAT}")
    except Exception as e:
        print(f"  ⚠️  Heatmap failed: {e}")
    
    # Dotplot
    print(f"\nGenerating dotplot (top 3 per cluster)...")
    try:
        fig = sc.pl.rank_genes_groups_dotplot(
            adata,
            n_genes=3,
            key='rank_genes_wilcox',
            use_raw=True,
            show=False,
            figsize=(14, 6)
        )
        plt.savefig(fig_dir / f'07_marker_dotplot.{FIGURE_FORMAT}', dpi=FIGURE_DPI, bbox_inches='tight')
        plt.show()
        print(f"  ✓ Saved: 07_marker_dotplot.{FIGURE_FORMAT}")
    except Exception as e:
        print(f"  ⚠️  Dotplot failed: {e}")

print("\n✓ Visualization complete")

## Step 12: Save Results

In [ ]:
# ============================================================================
# STEP 12: SAVE PROCESSED DATA
# ============================================================================

print("\n" + "="*80)
print("STEP 12: SAVING PROCESSED DATA")
print("="*80)

output_h5ad = output_dir / "tcell_bbknn_filtered_processed.h5ad"

print(f"\nSaving AnnData object...")
print(f"  Output: {output_h5ad}")
print(f"  Contents:")
print(f"    - Cells: {adata.n_obs:,}")
print(f"    - HVGs: {adata.n_vars:,}")
print(f"    - Full genes in .raw: {adata.raw.n_vars:,}")
print(f"    - BBKNN neighbors: ✓")
print(f"    - UMAP: ✓")
print(f"    - Leiden clusters: ✓")
print(f"    - Wilcoxon DE: ✓")
print(f"    - Epithelial filtered: ✓")

adata.write_h5ad(output_h5ad, compression='gzip')
print(f"\n✓ Data saved successfully")

## Summary

In [ ]:
# ============================================================================
# ANALYSIS SUMMARY
# ============================================================================

print("\n" + "="*80)
print("ANALYSIS COMPLETE")
print("="*80)

print(f"\nOutput directory: {output_dir}")

print(f"\nGenerated files:")
print(f"  ├── tcell_bbknn_filtered_processed.h5ad")
print(f"  ├── tcell_markers_wilcox_all.csv")
print(f"  ├── tcell_markers_wilcox_top10.csv")
print(f"  └── figures/")
print(f"      ├── 01_umap_clusters.{FIGURE_FORMAT}")
print(f"      ├── 02_umap_batch.{FIGURE_FORMAT}")
print(f"      ├── 03_umap_combined.{FIGURE_FORMAT}")
print(f"      ├── 04_tcell_markers_umap.{FIGURE_FORMAT}")
print(f"      ├── 05_tcell_markers_dotplot.{FIGURE_FORMAT}")
print(f"      ├── 06_marker_heatmap.{FIGURE_FORMAT}")
print(f"      └── 07_marker_dotplot.{FIGURE_FORMAT}")

if len(markers_df) > 0:
    print(f"\nMarker summary by cluster:")
    for cluster in sorted(markers_df['cluster'].unique()):
        cluster_markers = markers_df[markers_df['cluster'] == cluster]
        top3 = cluster_markers.head(3)['names'].tolist()
        print(f"  Cluster {cluster}: {len(cluster_markers)} markers")
        print(f"    Top 3: {', '.join(top3)}")

print("\n" + "="*80)
print("Key improvements in this run:")
print("  ✓ Epithelial contamination removed before analysis")
print("  ✓ Purer T/NK cell population for clustering")
print("  ✓ More accurate marker gene identification")
print("="*80)

print("\nNext steps:")
print("  1. Review canonical T cell markers (CD3D, CD4, CD8A, etc.)")
print("  2. Identify potential T cell subtypes:")
print("     - Naive T (CCR7+, SELL+, IL7R+)")
print("     - Memory T (GZMK+, CD69+)")
print("     - Effector T (GZMB+, PRF1+)")
print("     - Treg (FOXP3+, IL2RA+)")
print("     - NK (GNLY+, NKG7+)")
print("  3. Consider trajectory analysis for differentiation dynamics")
print("  4. Compare CD4+ vs CD8+ subsets if both present")
print("="*80)